## 数据规整：连接、联合和重塑

**长格式与宽格式**  
数据组织的两种不同方式，主要用于表格数据的表示和分析。  


  
**长格式（Long Format）**  


每个变量一列: 数据集中每个变量都占用一列，所有观测值（行）都保存在这些列中。  
适合重复测量: 当同一组数据在多个时间点或条件下进行观察时，长格式非常适用。  
更灵活: 长格式便于执行分组分析和汇总统计，许多数据分析和可视化工具（如 ggplot 和 seaborn）更容易处理这种格式。  

![jupyter](8.23.png)


**宽格式（Wide Format）**


每个分类变量一列: 每个分类变量（如 item）的值变成不同的列。  
易于比较: 宽格式让数据看起来更整齐，有利于快速查看和比较不同分类的值。  
使用特定工具: 一些数据处理工具或方法可能更适合处理宽格式数据，比如某些类型的透视表。  

![jupyter](8.24.png)


**何时使用长格式和宽格式**



使用长格式的场景  
统计分析: 当进行回归分析、方差分析或其他统计模型时，长格式更适合。  
可视化: 使用可视化工具（如 ggplot）进行图形展示时，通常需要长格式数据。  



使用宽格式的场景  
报表和演示: 如果数据需要以表格形式展示给观众，宽格式可能更清晰。  
数据比较: 在某些情况下，比较不同分类的数据时，宽格式可以更直观。  



**总结**   
长格式: 每个变量一列，适合分析和可视化，特别是重复测量的数据。   
宽格式: 每个分类变量一列，适合快速比较和表格展示。  

### 8.1 层次化索引  

层次化索引是pandas的一项重要功能，它使你能在一个轴上拥有多个（两个以上）索引
层级。

用另一种说法，它使你能以低维度形式处理高维度数据。

In [4]:
import numpy as np
import pandas as pd
import random

In [23]:
data = pd.Series(np.random.uniform(size=9),
                 index=[["a", "a", "a", "b", "b", "c", "c", "d", "d", ],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data

a  1    0.826182
   2    0.261594
   3    0.032080
b  1    0.532014
   3    0.990004
c  1    0.909482
   2    0.858583
d  2    0.114510
   3    0.050834
dtype: float64

看到的结果是以MuLtiIndex作为索引的经过美化的Series视图。索引之间的"间隔"
表示"直接使用上面的标签"：

In [25]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

In [27]:
# 对于层次化索引对象，可以使用部分索引，使用它选取数据子集更为简单：
data["b"]

1    0.532014
3    0.990004
dtype: float64

In [29]:
data["b":"c"]    #切片，索引标签的切片，而不是基于位置的切片。（包括这两个标签所对应的行）

b  1    0.532014
   3    0.990004
c  1    0.909482
   2    0.858583
dtype: float64

从 DataFrame 或 Series 中选取基于索引标签的行切片，包含起始和结束标签所对应的行。

In [31]:
data.loc[["b", "d"]]    #通过标签选取特定行，loc是基于标签的选取方法，选取多列、行和多标签

b  1    0.532014
   3    0.990004
d  2    0.114510
   3    0.050834
dtype: float64

In [33]:
data.iloc[:]    #iloc 是基于位置的选择，里面的项只能是数值，且是半开区间

a  1    0.826182
   2    0.261594
   3    0.032080
b  1    0.532014
   3    0.990004
c  1    0.909482
   2    0.858583
d  2    0.114510
   3    0.050834
dtype: float64

loc：基于标签的选择，使用行标签和列标签进行数据选择。切片时包含结束标签。  
iloc：基于位置的选择，使用行和列的位置进行数据选择。切片时不包含结束位置。  


如果你知道行或列的标签，使用 loc。  
如果你知道行或列的位置，使用 iloc。  

In [36]:
# 在“内部”层级中进行选取也是可行的。选取第二层索引值为2的数据：
data.loc[:, 2]    #所有index标签，然后再选取第二层级含有 2 的行

a    0.261594
c    0.858583
d    0.114510
dtype: float64

In [40]:
data.loc[:, :2]  #所有index标签，选取每个标签的前两行

a  1    0.826182
   2    0.261594
b  1    0.532014
c  1    0.909482
   2    0.858583
d  2    0.114510
dtype: float64

层次化索引在数据重塑和基于分组的操作（如生成透视表）中发挥着关键的作用。  

In [42]:
# 通过unstack方法将下面的数据重排到DataFrame中：
data.unstack()    #将刚才的data以 DF 形式展示出来

,1,2,3
a,0.826182,0.261594,0.032080
b,0.532014,NaN,0.990004
c,0.909482,0.858583,NaN
d,NaN,0.114510,0.050834


还有另一种转化DF的方式：str.extract方法可以用DataFrame的形式返回正则表达式获取的分组：   

因为标签b中索引2没有值，所以为空，以此类推

In [19]:
# unstack 的逆运算是 stack
data.unstack().stack()

a  1    0.640352
   2    0.845137
   3    0.115763
b  1    0.935888
   3    0.210510
c  1    0.983957
   2    0.358978
d  2    0.673563
   3    0.457544
dtype: float64

In [44]:
# 对于DataFrame，每个轴都可以有分层索引：
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[["a", "a", "b", "b"], [1, 2, 1, 2]],  #行标签
                     columns=[["Ohio", "Ohio", "Colorado"],    #列标签
                              ["Green", "Red", "Green"]])
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

In [52]:
#各层都可以有名称（可以是字符串，也可以是任意Python对象）。
#如果指定了名称，它们就会显示在控制台输出中：

frame.index.names = ["key1", "key2"]

frame.columns.names = ["state", "color"]
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

分层索引的名称取代了name属性，后者只用于单层索引l。

这里需要注意，索引名“state”和“color”不属于行标签（即frame, index的值）。



In [54]:
# 通过 nlevels 属性，可以知道索引有多少层：
frame.index.nlevels

2

In [56]:
#使用部分列索引，选取列分组：
frame["Ohio"]

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

In [58]:
#可以单独创建MultiIndex，然后复用。前面DataFrame中的列带有层级名称，还可以如下创建：
pd.MultiIndex.from_arrays([["Ohio", "Ohio", "Colorado"],
                           ["Green", "Red", "Green"]],
                           names=["state", "color"])

MultiIndex([(    'Ohio', 'Green'),
            (    'Ohio',   'Red'),
            ('Colorado', 'Green')],
           names=['state', 'color'])

MultiIndex 是 pandas 中支持分层索引的数据结构，允许你为 DataFrame 或 Series 创建多个级别的索引，适用于处理更复杂的数据集。它可以让你通过多个层次对数据进行访问和操作，类似于在二维的基础上增加更多维度。  

层次化数据：特别适用于时间序列、金融数据等场景，多个层次能更有效地管理和查询数据。  
交叉表：通过 MultiIndex 可以很方便地生成和处理交叉表数据（比如，按行和列分层统计）。  


可以通过 .unstack() 方法将 MultiIndex 展开为普通 DataFrame，或者通过 stack() 把普通 DataFrame 压缩为 MultiIndex。  

#### 8.1.1 重排序和层级排序  
调整某条轴上各层级的顺序或根据指定层级上的值对数据进行排序。  

In [60]:
#swaplevel方法接收两个层级编号或名称，并返回一个层级互换的新对象（但数不会发生变化）：
frame_swap = frame.swaplevel("key1", "key2")
frame_swap

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

虽然 key1 和 key2 已经交换了层级，但索引顺序保持不变。  
为了按照新的层级顺序进行排序，我们需要使用 sort_index()：  

In [62]:
frame_sorted = frame_swap.sort_index()
frame_sorted

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

key1 和 key2 通过索引排序后，将 1， 1 和 2，2 转化为一个 index 标签折叠起来

sort_index默认根据所有索引层级中的字母顺序对数据进行排序，  


也可以通过传人 level 参数只选取单层级或层级的子集

In [66]:
frame.sort_index(level=1)    #frame按 key2 列排序，不折叠是MultiIndex默认行为

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

In [68]:
frame.swaplevel(0, 1).sort_index(level=0)   #交换行索引第一层和第二层，按交换后的第一层排序

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

In [70]:
frame.swaplevel(0, 1).sort_index(level=1)   #第二index就是不折叠的

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

为什么不是操作列索引（columns）？  


Pandas 的 swaplevel 和 sort_index 默认操作的是 行索引，而不是 列索引。 


如果你想对列索引进行操作，你需要明确指定使用 axis=1。  


例如，要交换列索引的级别，你可以这样做：  

In [72]:
frame.swaplevel(0, 1, axis=1).sort_index(level=0, axis=1)  #交换后按 color列排序

color        Green       Red
state     Colorado Ohio Ohio
key1 key2                   
a    1           2    0    1
     2           5    3    4
b    1           8    6    7
     2          11    9   10

color 中先是green，然后是red，green中有两个，下面按state排序，colorado 在 ohio前面，最后就是这么个排序方式

In [126]:
frame.swaplevel(0, 1, axis=1).sort_index(level=1, axis=1)

color        Green       Red
state     Colorado Ohio Ohio
key1 key2                   
a    1           2    0    1
     2           5    3    4
b    1           8    6    7
     2          11    9   10

同理，按照 state 排序，先是 colorado， 然后是ohio，因为ohio有两个，一个green一个red，先green再red，所以交换了1、3列

#### 8.1.2 按层级进行汇总统计  
许多对DataFrame和Series的描述性和汇总性统计都有一个Level选项，用于指定在某条轴的特定层级进行聚合。  

In [139]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [57]:
# groupby 按照 MultiIndex 中指定的层级（key2）对数据进行分组。
# key2 是第二层索引，并且对分组后的数据进行做合
frame.groupby(level="key2").sum()    #key2 有两个组，一个是1，一个是2，就把两个组中的数值相加

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

key1 默认是被丢弃的，想要不丢弃key1，可以用reset_index() 或者使用 as_index=False 来保留原来的索引结构。

In [68]:
frame.groupby("key2").sum().reset_index()  #仍不显示。应该是默认就把key1删除了

state key2  Ohio     Colorado
color      Green Red    Green
0        1     6   8       10
1        2    12  14       16

In [70]:
frame.groupby(level="color", axis="columns").sum()

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_2624\775557097.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  frame.groupby(level="color", axis="columns").sum()


color      Green  Red
key1 key2            
a    1         2    1
     2         8    4
b    1        14    7
     2        20   10

这个警告提示你 DataFrame.groupby 函数在 axis=1（即按列进行分组）时已被弃用。


在未来的 Pandas 版本中，你应该使用 .T.groupby(...) 来替代。

In [154]:
frame_T = frame.T
frame_T

key1            a     b    
key2            1  2  1   2
state    color             
Ohio     Green  0  3  6   9
         Red    1  4  7  10
Colorado Green  2  5  8  11

In [160]:
reslut = frame_T.groupby(level="color").sum()  #现在就将color行中可以合并的都合并了
reslut

key1   a      b    
key2   1  2   1   2
color              
Green  2  8  14  20
Red    1  4   7  10

In [166]:
reslut.T   #与前面的frame.groupby(level="color", axis="columns").sum() 效果相同

color      Green  Red
key1 key2            
a    1         2    1
     2         8    4
b    1        14    7
     2        20   10

解析：将Green列的数据全部相加，

![jupyter](8.1.png)
![jupyter](8.2.png)



green 列，0+2  3+5 6+8  9+11

#### 8.1.3 使用 DF 的列进行索引  


In [74]:
#通常不会将DF的单列或多列用作行索引，但是可能将行索引用作DF的列：
frame = pd.DataFrame({"a": range(7), "b": range(7, 0, -1),
                      "c": ["one", "one", "one", "two", "two",
                            "two", "two"],
                      "d": [0, 1, 2, 0, 1, 2, 3]})
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


In [76]:
frame2 = frame.set_index(["d", "c"])     #将c、d列设为索引，谁在列表前谁默认为第一列
frame2

,,a,b
d,c,,
0,one,0,7
1,one,1,6
2,one,2,5
0,two,3,4
1,two,4,3
2,two,5,2
3,two,6,1


In [78]:
#DF 的set_index函数会将单列或多列转换为行索引，并创建一个新的DF:
frame2 = frame.set_index(["c", "d"])     #将c、d列设为索引
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

In [80]:
#默认情况下，这些列（c、d）列会从DF中移除，但也可以通过传入drop=False将其保留下来：
frame.set_index(["c", "d"], drop=False)

a  b    c  d
c   d              
one 0  0  7  one  0
    1  1  6  one  1
    2  2  5  one  2
two 0  3  4  two  0
    1  4  3  two  1
    2  5  2  two  2
    3  6  1  two  3

In [205]:
# reset_index的功能与set_index相反，它将层次化索引的层级转移到列：
frame2.reset_index()     #将索引转化为普通列

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1


将当前的索引转为普通的列，并生成一个新的整数索引，如果你不希望原索引变为列，可以设置 drop=True

In [212]:
frame2.reset_index(drop=True)

,a,b
0,0,7
1,1,6
2,2,5
3,3,4
4,4,3
5,5,2
6,6,1


### 8.2 联合与合并数据集  
pandas.merge  

可根据单个或多个键将不同DataFrame中的行连接起来。  
SQL或其他关系型数据库的用户对此应该会比较熟悉，因为它实现的就是数据库的join（连接）操作。


pandas.concat  

沿一条轴将多个对象连接或“堆叠”到一起。


combine_first  

将重复数据拼接在一起，用一个对象中的值填充另一个对象中的缺失值。

#### 8.2.1 数据库风格的 DF 连接  
数据集的合并或连接运算是通过单个或多个键将行连接起来的。  
这些运算对于（基于SQL
的）关系型数据库非常重要。

In [82]:
# pandas主要使用pandas.merge函数对数据执行连接操作。
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64")})

df2 = pd.DataFrame({"key": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [84]:
df2

,key,data2
0,a,0
1,b,1
2,d,2


In [86]:
# 这是多对一连接的示例。df1中的数据有多行的标签为a和b，
# df2中key列的每个值则仅对应一行。对这些对象调用pandas.merge即可得到：
pd.merge(df1, df2)

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


在 pd.merge(df1, df2) 时，c 行被丢掉是因为 merge 操作默认是进行内连接（inner join）。


内连接只会保留两个 DataFrame 中 key 列都有的值，而 c 只在 df1 中出现，不在 df2 中，因此这行数据不会被保留下来。  


要保留 df1 中所有的行，即使它们在 df2 中没有对应的 key，你可以使用左连接（left join）：

In [89]:
pd.merge(df1, df2, how="left")

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,c,3,<NA>
4,a,4,0
5,a,5,0
6,b,6,1


In [91]:
# 不指定用哪个列进行连接pd.merge就会将重叠的列名当作键，最好还是指定：
pd.merge(df1, df2, on="key", how="left")

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,c,3,<NA>
4,a,4,0
5,a,5,0
6,b,6,1


In [93]:
#pd.merge输出结果的列顺序是未指定的,如果两个对象的列名不同，也可以分别进行指定：
df3 = pd.DataFrame({"lkey": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64")})
df4 = pd.DataFrame({"rkey": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})
df3

,lkey,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [97]:
df4

,rkey,data2
0,a,0
1,b,1
2,d,2


In [99]:
pd.merge(df3, df4, left_on="lkey", right_on="rkey")

,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,a,2,a,0
3,a,4,a,0
4,a,5,a,0
5,b,6,b,1


left_on 和 right_on 参数用于指定两个 DataFrame 中的不同列作为合并的键。  

具体解释：  

left_on="lkey"：表示在左边的 DataFrame（即 df3）中使用 lkey 列作为合并的依据。  
right_on="rkey"：表示在右边的 DataFrame（即 df4）中使用 rkey 列作为合并的依据。  
这意味着 pandas 会按照 df3 的 lkey 列与 df4 的 rkey 列进行对比，并根据匹配结果合并这两个 DataFrame。  

In [101]:
# 在merge时，c，d因为不是共同键消失了，因为默认是内连接，结果是键的交集
# 可以用左连接、右连接、外连接，外连接是取并集，融合了左右连接的效果
pd.merge(df1, df2, how="outer")

,key,data1,data2
0,a,2,0
1,a,4,0
2,a,5,0
3,b,0,1
4,b,1,1
5,b,6,1
6,c,3,<NA>
7,d,<NA>,2


In [103]:
pd.merge(df3, df4, left_on="lkey", right_on="rkey", how="outer")

,lkey,data1,rkey,data2
0,a,2,a,0
1,a,4,a,0
2,a,5,a,0
3,b,0,b,1
4,b,1,b,1
5,b,6,b,1
6,c,3,NaN,<NA>
7,NaN,<NA>,d,2


左连接（left join）取左边 DataFrame 中的所有元素，即使右边的 DataFrame 中有更多的元素，也不会影响左边的结果。如果右边的 DataFrame 中某些键在左边 DataFrame 中找不到对应的行，那么这些右边 DataFrame 中的行将被忽略。



左连接会保留左边 DataFrame 中的所有行，并从右边 DataFrame 中查找匹配的行。如果右边 DataFrame 中没有找到匹配的行，则对应的右边列会填充 NaN。

右连接以此类推

在外连接中，如果左侧或右侧DataFrame对象的行与其他DataFrame中的键不匹配，则
这些不匹配的行将以NA值的方式出现在其他DataFrame的列中。

how 参数不同的连接类型  
![jupyter](8.3.png)

In [107]:
# 多对多的合并会生成匹配键的笛卡儿积
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"],
                    "data1": pd.Series(range(6), dtype="Int64")})
df2 = pd.DataFrame({"key": ["a", "b", "a", "b", "d"],
                    "data2": pd.Series(range(5), dtype="Int64")})
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [109]:
df2

,key,data2
0,a,0
1,b,1
2,a,2
3,b,3
4,d,4


In [112]:
#内连接取交集：由于左边的DataFrame有3个"b”行，右边的DataFrame有2个"b”行，
#所以最终结果中就有6个"b”行。how关键字的连接方式仅影响出现在结果中的不同键值：
pd.merge(df1, df2, how="inner")

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,a,4,0
7,a,4,2
8,b,5,1
9,b,5,3


内连接的结果是只保留两个 DataFrame 中键值（key 列）的交集部分，即两个 DataFrame 中都存在的 key 值的匹配行。  
它不会生成所有可能的组合，而只是找出匹配的键值并将它们合并在一起。  
键值的交集：  

df1 的 key 列中有 "b", "b", "a", "c", "a", "b"。  
df2 的 key 列中有 "a", "b", "a", "b", "d"。  
交集是 "a" 和 "b"。  
合并：  

对于每个交集中的键值，df1 和 df2 中所有匹配的行都会被合并在一起。  
所以最后只输出了有的 a 和 b，其余的全部都丢弃

比如，key data1 第一行为 b 0 ，第二个DF中有两个b，所以 data1 的第一行要组合两次与data2中的b，组合为 b 0 1 ，b 0 3

In [111]:
# 根据多个键进行和并，传入一个由列名组成的列表即可
left = pd.DataFrame({"key1": ["foo", "foo", "bar"],
                     "key2": ["one", "two", "one"],
                     "lval": pd.Series([1, 2, 3], dtype='Int64')})
right = pd.DataFrame({"key1": ["foo", "foo", "bar", "bar"],
                      "key2": ["one", "one", "one", "two"],
                      "rval": pd.Series([4, 5, 6, 7], dtype='Int64')})
left

,key1,key2,lval
0,foo,one,1
1,foo,two,2
2,bar,one,3


In [151]:
right

,key1,key2,rval
0,foo,one,4
1,foo,one,5
2,bar,one,6
3,bar,two,7


In [153]:
pd.merge(left, right, on=["key1", "key2"], how="outer")

,key1,key2,lval,rval
0,bar,one,3,6
1,bar,two,<NA>,7
2,foo,one,1,4
3,foo,one,1,5
4,foo,two,2,<NA>


外连接取并集，先列举出所有可能的情况，再去合并  


比如 left 中的 foo one 1， 在right 有两个，foo one，那么 1 和right中所有的数都要组合一次

默认排序应该是key1

In [117]:
# 测试 重复出现外连接
left1 = pd.DataFrame({"key1": ["foo", "foo", "bar", "foo"],
                     "key2": ["one", "two", "one", "one"],
                     "lval": pd.Series([1, 2, 3, 5], dtype='Int64')})
right1 = pd.DataFrame({"key1": ["foo", "foo", "bar", "bar"],
                      "key2": ["one", "one", "one", "two"],
                      "rval": pd.Series([5, 6, 7, 8], dtype='Int64')})
left1

,key1,key2,lval
0,foo,one,1
1,foo,two,2
2,bar,one,3
3,foo,one,5


In [119]:
right1

,key1,key2,rval
0,foo,one,5
1,foo,one,6
2,bar,one,7
3,bar,two,8


In [121]:
pd.merge(left1, right1, on=["key1", "key2"], how="outer")

,key1,key2,lval,rval
0,bar,one,3,7
1,bar,two,<NA>,8
2,foo,one,1,5
3,foo,one,1,6
4,foo,one,5,5
5,foo,one,5,6
6,foo,two,2,<NA>


当 foo one 在两个DF中都出现两次后，就会点积的形式去组合，DF1 有两个，DF2也有两个，最后结果是2 * 2 =4个  

即使后面的数值是一样的也不影响，比如DF1有foo one 5， DF2也有foo one 5，后面合并就是foo one 5 5，不影响

在进行列-列连接时，会丢弃传人DataFrame对象中的索引。如果需要保留
索引值，可以使用reset_index将索引追加到列。

In [171]:
left

,key1,key2,lval
0,foo,one,1
1,foo,two,2
2,bar,one,3


In [173]:
right

,key1,key2,rval
0,foo,one,4
1,foo,one,5
2,bar,one,6
3,bar,two,7


foo 在DF 中一共有四个，DF1 中，foo one 1 是一种组合，对应下面 one 4 ， one 5，都可以组合，向下以此类推，最后只有四个

需要注意的就是 one 1 、 one 4 是一体的

In [169]:
# 合并操作中重复列名的处理
pd.merge(left, right, on="key1")

,key1,key2_x,lval,key2_y,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


pd默认给相同列名的列加上了区分的_x 和_y， 除了手工处理，添加前缀，可以用 suffixes选项，可以对指定重叠名添加需要的字符串,默认为(_x, _y)


合并情况：  
key1 有 foo 和 bar 两个关键字，所以要左右两个表的 foo、bar 相互对应，左表 foo 开始，foo one 1，右表 foo 的有 foo one 4， foo one 5，左表第一次匹配结束  


第二行开始， 左表 foo two 2 对应的右表有 foo 的行有： foo one 4， foo one 5，含有 foo 关键字的对应完毕  


第三行开始， 左表开始关键字 bar， 完整对应的是 bar one 3， 右表含有关键字 bar 的有 bar one 6， bar two 7，组合完毕



In [126]:
pd.merge(left, right, on="key1", suffixes=("_left", "_right"))    #suffixes加后缀

,key1,key2_left,lval,key2_right,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


#### pd.merge参数

![jupyter](8.4.png)
![jupyter](8.5.png)

#### 根据索引合并  
在某些情况下，DataFrame中的连接键位于其索引（行标签）中。在这种情况下，可以传
入left_index=True或right_index=True（也可以两个都传）以说明将索引用作连
接键：

In [128]:
left2 = pd.DataFrame({"key": ["a", "b", "a", "a", "b", "c"],
                      "value": pd.Series(range(6), dtype="Int64")})
right2 = pd.DataFrame({"group_val": [3.5, 7]}, index=["a", "b"])
left2

,key,value
0,a,0
1,b,1
2,a,2
3,a,3
4,b,4
5,c,5


In [130]:
right2

,group_val
a,3.5
b,7.0


In [209]:
# 左边使用 'key' 列，右边使用索引进行匹配
pd.merge(left2, right2, left_on="key", right_index=True)

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0


将 left2 和 right2 进行合并，其中 left2 使用其 key 列作为连接键，而 right2 使用其索引（index）作为连接键。  


丢弃了不在 right2 索引中的 c，前面三个 a，两个 b 都与right中的 group_val中的值合并了


说白了就是将group_val 中的a=3.5赋值给left2中，a 0，a 2，a 3都是3.5，b也是

In [213]:
# 求取并集
pd.merge(left2, right2, left_on="key", right_index=True, how="outer")

,key,value,group_val
0,a,0,3.5
2,a,2,3.5
3,a,3,3.5
1,b,1,7.0
4,b,4,7.0
5,c,5,NaN


在group_val中没有 c 值，所以取并集后 c 的group_val为Na


![jupyter](8.6.png)

In [218]:
# 对于层次化索引的数据，复杂很多，索引连接实际上是多键合并
lefth = pd.DataFrame({"key1": ["Ohio", "Ohio", "Ohio",
                               "Nevada", "Nevada"],
                      "key2": [2000, 2001, 2002, 2001, 2002],
                      "data": pd.Series(range(5), dtype="Int64")})
right_index = pd.MultiIndex.from_arrays(
    [
        ["Nevada", "Nevada", "Ohio", "Ohio", "Ohio", "Ohio"],
        [2001, 2000, 2000, 2000, 2001, 2002]
    ]
)
righth = pd.DataFrame({"event1": pd.Series([0, 2, 4, 6, 8, 10], dtype="Int64", index=right_index),
                       "event2": pd.Series([1, 3, 5, 7, 9, 11], dtype="Int64", index=right_index)})

lefth

,key1,key2,data
0,Ohio,2000,0
1,Ohio,2001,1
2,Ohio,2002,2
3,Nevada,2001,3
4,Nevada,2002,4


In [220]:
righth

event1  event2
Nevada 2001       0       1
       2000       2       3
Ohio   2000       4       5
       2000       6       7
       2001       8       9
       2002      10      11

In [225]:
# 必须以列表的形式指明用作合并键的多个列
#多重索引作为合并对象时，会根据多重索引的所有层进行匹配
pd.merge(lefth, righth, left_on=["key1", "key2"], right_index=True)

,key1,key2,data,event1,event2
0,Ohio,2000,0,4,5
0,Ohio,2000,0,6,7
1,Ohio,2001,1,8,9
2,Ohio,2002,2,10,11
3,Nevada,2001,3,0,1


默认取交集，lefth 中 ohio 2000 对应 righth 中有两个 ohio 2000， 所以最后应合并出两个ohio 2000，
其余的都是一个，lefth 有 Nevada 2002，而righth没有，则不会合并这一条记录，因为是取交集

![jupyter](8.7.png)

多重索引作为合并对象时，会根据多重索引的所有层进行匹配，如果想指定特定的层，需要设置level

In [230]:
#外连接
pd.merge(lefth, righth, left_on=["key1", "key2"], right_index=True, how="outer")

,key1,key2,data,event1,event2
4,Nevada,2000,<NA>,2,3
3,Nevada,2001,3,0,1
4,Nevada,2002,4,<NA>,<NA>
0,Ohio,2000,0,4,5
0,Ohio,2000,0,6,7
1,Ohio,2001,1,8,9
2,Ohio,2002,2,10,11


此时，之前inner没有出现的记录也被加了进来

In [233]:
# 同时使用双方的索引也没问题：
left3 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]], index=["a", "c", "e"],
                     columns=["Ohio", "Nevada"]).astype("Int64")
right3 =pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]], 
                     index=["b", "c", "d", "e"],
                     columns=["Missouri", "Alabama"]).astype("Int64")
left3

,Ohio,Nevada
a,1,2
c,3,4
e,5,6


In [235]:
right3

,Missouri,Alabama
b,7,8
c,9,10
d,11,12
e,13,14


In [239]:
# 双索引合并
pd.merge(left3, right3, left_index=True, right_index=True, how="outer")

,Ohio,Nevada,Missouri,Alabama
a,1,2,<NA>,<NA>
b,<NA>,<NA>,7,8
c,3,4,9,10
d,<NA>,<NA>,11,12
e,5,6,13,14


通过外连接，left 中有c、e 与right是重叠的，所以c、e行没有缺失值，其余的要么 lfte 缺，要么right 缺

![jupyter](8.8.png)

In [242]:
# DF还可以用 join 快速实现按索引合并，可以快速合并带有相同或相似的索引的DF，
# 但索引不能是多层的
left3.join(right3, how="outer")

,Ohio,Nevada,Missouri,Alabama
a,1,2,<NA>,<NA>
b,<NA>,<NA>,7,8
c,3,4,9,10
d,<NA>,<NA>,11,12
e,5,6,13,14


效果与上述pd.merge效果相同  


join 默认是左连接，所以当连接的数据行数较多时，会直接丢弃多的数据不做连接

In [132]:
# 与pd.merge相比，DF的join方法默认是左连接，并且还支持在调用的 DF 的列上连接传入 DF的索引
# 人话： join默认是索引做合并，而merge默认用指定的键合并，所以不给join任何值，他就会用索引去合并
# join默认是左连接****
left2.join(right2, on="key")

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0
5,c,5,NaN


join默认是索引做合并，而merge默认用指定的键合并，所以不给join任何值，他就会用索引去合并  
join做左连接时，右边没有左边的值会补为NA

![jupyter](8.9.png)

In [250]:
#对于简单的索引合并，可以向join传入一组DataFrame，pandas.concat函数，它也能实现此功能：
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=["a", "c", "e", "f"],
                       columns=["New York", "Oregon"])
another

,New York,Oregon
a,7.0,8.0
c,9.0,10.0
e,11.0,12.0
f,16.0,17.0


In [252]:
left3.join([right2, another])

,Ohio,Nevada,group_val,New York,Oregon
a,1,2,3.5,7.0,8.0
c,3,4,NaN,9.0,10.0
e,5,6,NaN,11.0,12.0


因为是left3做左连接，left3只有 a、c、e三行，所以不管right3 和 another有再多行也不会添加进去，只会合并其中的a、c、e的值

![jupyter](8.10.png)

In [257]:
left3.join([right3, another], how="outer")

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1,2,<NA>,<NA>,7.0,8.0
c,3,4,9,10,9.0,10.0
e,5,6,13,14,11.0,12.0
b,<NA>,<NA>,7,8,NaN,NaN
d,<NA>,<NA>,11,12,NaN,NaN
f,<NA>,<NA>,<NA>,<NA>,16.0,17.0


#### 8.2.3 轴向拼接
又叫做拼接、堆叠

In [262]:
# concatenate函数可以实现对Np数组的拼接：
arr = np.arange(12).reshape((3, 4))
arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [268]:
np.concatenate([arr, arr])   #行拼接

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [270]:
np.concatenate([arr, arr], axis=1)  #列拼接

array([[ 0,  1,  2,  3,  0,  1,  2,  3],
       [ 4,  5,  6,  7,  4,  5,  6,  7],
       [ 8,  9, 10, 11,  8,  9, 10, 11]])

对于pd对象(Series、DF)，带有标签的轴能进一步泛化数组的拼接操作。  

还有以下几个注意事项：  

如果对象在其他轴上的索引不同，应该是合并轴上的不同元素还是只使用交集？  

拼接的数据集是否需要在结果对象中可识别？  

拼接轴中的数据是否需要保存？  DF默认整数标签在拼接时删除接时删掉。

In [156]:
# pandas的concat函数提供了解决这些问题的一致方式
s1 = pd.Series([0, 1], index=["a", "b"], dtype="Int64")
s2 = pd.Series([2, 3, 4], index=["c", "d", "e"], dtype="Int64")
s3 = pd.Series([5, 6], index=["f", "g"], dtype="Int64")

In [140]:
s1

a    0
b    1
dtype: Int64

In [142]:
s2

c    2
d    3
e    4
dtype: Int64

In [144]:
s3

f    5
g    6
dtype: Int64

In [146]:
# 将每个 Series 的索引和值按顺序拼接在一起
pd.concat([s1, s2, s3])   #默认按axis=0的方向合并，行方向

a    0
b    1
c    2
d    3
e    4
f    5
g    6
dtype: Int64

默认情况下，pandas.concat是在axis="index”上工作的，最终生成一个新Series。如果传入axis="columns"，则结果就会变成DataFrame：

In [149]:
pd.concat([s1, s2, s3], axis=1)    #s1,s2,s3共三列

,0,1,2
a,0,<NA>,<NA>
b,1,<NA>,<NA>
c,<NA>,2,<NA>
d,<NA>,3,<NA>
e,<NA>,4,<NA>
f,<NA>,<NA>,5
g,<NA>,<NA>,6


pd.concat 沿着列的方向拼接数据，每个 Series 都成为 DataFrame 的一列。 


首先要做的是 `索引对齐` 。当多个 Series 或 DataFrame 被沿着列方向拼接（即 axis=1）时，它会根据索引对齐各个对象，将相同索引的数据放在同一行，缺失的部分用 NaN 填充。

s1第一列，只有值0，1  
s2第二列，有2，3，4.  
s3第三列，只有5， 6  


s1、s2、s3 的索引被组合起来，结果 DataFrame 的索引是三个 Series 的所有唯一索引值的并集（a, b, c, d, e, f, g）。  


没有对应索引的行会用 NaN 来填充。


结果 DataFrame 中的列没有手动指定名称，因此 Pandas 自动给它们分配了列名为 0、1、2（分别对应 s1, s2, s3）。   


如果你想手动指定列名，可以使用 keys 参数来提供列名。

In [174]:
t1 = pd.Series([0, 1, 2], index=["a", "b", "c"], dtype="Int64")
t1

a    0
b    1
c    2
dtype: Int64

In [162]:
t2 = pd.concat([t1, s2], axis=1)
t2

,0,1
a,0,<NA>
b,1,<NA>
c,2,2
d,<NA>,3
e,<NA>,4


此时，t1 和 s2 有共同的行 c 2,进行列合并时，先对所有索引进行排序，每个Series 或 DF 是一列，然后每列对有的数据进行填充。

In [152]:
# 另外的轴上没有重叠，实际上就是索引的并集（即外连接）。
# 传入join="inner”即可得到它们的交集：
s4 = pd.concat([s1, s3])
s4

a    0
b    1
f    5
g    6
dtype: Int64

In [154]:
pd.concat([s1, s4], axis=1)    #默认外连接, 取并集

,0,1
a,0,0
b,1,1
f,<NA>,5
g,<NA>,6


因为 s4 是 s1 和 s3 合并的，所以 s4 中也有 a 0，b 1，沿列合并时就s1 没有f，g，s4 都有

In [317]:
pd.concat([s1, s4], axis=1, join="inner")   #取交集

,0,1
a,0,0
b,1,1


因为使用的是join="inner"，所以"f”和"g”标签消失了。

另一个问题是在结果中无法辨认参与拼接的数据。

In [321]:
# 在拼接轴上创建层次化索引，使用参数keys来实现：
result = pd.concat([s1, s1, s3], keys=["one", "two", "three"])
result

one    a    0
       b    1
two    a    0
       b    1
three  f    5
       g    6
dtype: Int64

pd.concat() 中，keys 参数的名称是固定的，不能替换成其他单词。keys 是用于给拼接结果增加一个外层索引的特定参数，虽然你不能修改这个参数名，但你可以传入任何自定义的标签（如字符串、数字等）来标识不同的被拼接对象。

分别在进行拼接的 s1, s1, s3 上层添加了一个行标签

In [330]:
result.unstack()    #转化为DF输出

,a,b,f,g
one,0,1,<NA>,<NA>
two,0,1,<NA>,<NA>
three,<NA>,<NA>,5,6


unstack() 默认将最内层的索引转换为列，但你可以通过参数指定要转换的层。    



对多级索引的 DataFrame，使用 unstack() 可以轻松地重新排列数据的表示形式。

In [332]:
# 沿着axis="columns”对Series进行合并，keys则会成为DataFrame的列标签：
pd.concat([s1, s2, s3], axis=1, keys=["one", "two", "three"])

,one,two,three
a,0,<NA>,<NA>
b,1,<NA>,<NA>
c,<NA>,2,<NA>
d,<NA>,3,<NA>
e,<NA>,4,<NA>
f,<NA>,<NA>,5
g,<NA>,<NA>,6


因为按列合并的话每一个Series都会转化为DF的一个列，所以添加列标签的话直接就是在新DF上的列添加即可

In [370]:
# 对于Series 的操作都适用于DF
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), index=["a", "b", "c"],
                   columns=["one", "two"])
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), index=["a", "c"],
                   columns=["three", "four"])

In [337]:
df1

,one,two
a,0,1
b,2,3
c,4,5


In [339]:
df2

,three,four
a,5,6
c,7,8


In [349]:
pd.concat([df1, df2], keys=["level1", "level2"])   #这里，参数keys被用来创建层次化索引，其中的第一级可以用于判断参与拼接的DataFrame对象。

one  two  three  four
level1 a  0.0  1.0    NaN   NaN
       b  2.0  3.0    NaN   NaN
       c  4.0  5.0    NaN   NaN
level2 a  NaN  NaN    5.0   6.0
       c  NaN  NaN    7.0   8.0

`与前面Series的按行合并不同，此时列有列名，再做合并的话，需要将各个列名先排列起来再去组合各个行`  
`Series是一维数据结构，不能添加列名`  



对于 DataFrame，合并时列会自动根据列名对齐，并生成多层索引。




keys 参数用于创建一个 分层索引（MultiIndex）。  


keys 会为每个被连接的 DataFrame 或 Series 分配一个 层级标签，这些标签会成为结果中外层索引的值。    


这样可以在合并后的结果中更清晰地区分不同的数据来源。  

In [341]:
pd.concat([df1, df2], axis=1, keys=["level1", "level2"])

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

列合并时，先把所有的行索引排列好，然后再添加数据

In [347]:
pd.concat([df1, df2], keys=["level1", "level2"], join="inner")

Empty DataFrame
Columns: []
Index: [(level1, a), (level1, b), (level1, c), (level2, a), (level2, c)]

In [358]:
# 如果传入的不是列表而是对象字典，则字典的键就会用于keys选项：
pd.concat({"level1": df1, "level2": df2}, axis=1)    #键作为分层索引

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

In [360]:
pd.concat({"level1": df1, "level2": df2})

one  two  three  four
level1 a  0.0  1.0    NaN   NaN
       b  2.0  3.0    NaN   NaN
       c  4.0  5.0    NaN   NaN
level2 a  NaN  NaN    5.0   6.0
       c  NaN  NaN    7.0   8.0

In [372]:
# 管理层次化索引创建方式的参数，names参数命名创建的轴层级
pd.concat([df1, df2], axis=1, keys=["level1", "level2"],
          names=["upper", "lower"])     #多级索引每一层的命名

upper level1     level2     
lower    one two  three four
a          0   1    5.0  6.0
b          2   3    NaN  NaN
c          4   5    7.0  8.0

names 参数用于为 多级索引（MultiIndex）的每一层命名。 

keys 是创建的外层索引。  
当使用 keys 参数创建了多级索引后，names 参数可以为这些索引层次提供名称，帮助更好地理解和区分不同层次的索引。



upper 是第一层索引，来自 keys=["level1", "level2"]，它代表你使用 keys 给合并的 DataFrame 分配的标签。在输出中，upper 对应的就是 level1（df1）和 level2（df2）。


lower 是第二层索引，代表各个 DataFrame 的列名。在输出中，lower 对应的就是 df1 的列名 one 和 two，以及 df2 的列名 three 和 four。

In [379]:
pd.concat([df1, df2], keys=["level1", "level2"],
          names=["upper", "lower"])     #多级索引命名

one  two  three  four
upper  lower                       
level1 a      0.0  1.0    NaN   NaN
       b      2.0  3.0    NaN   NaN
       c      4.0  5.0    NaN   NaN
level2 a      NaN  NaN    5.0   6.0
       c      NaN  NaN    7.0   8.0

upper：第一层索引，代表 keys=["level1", "level2"]。它区分了来自 df1 的行（用 level1 标记）和来自 df2 的行（用 level2 标记）。  



lower：第二层索引，代表原始的行索引（a, b, c），这些是 df1 和 df2 中的行标签。它们成为了多级索引的内层部分。


lower 这一层是原始的行索引 a, b, c。

In [389]:
df1 = pd.DataFrame(np.random.standard_normal((3, 4)),
                   columns=["a", "b", "c", "d"])
df2 = pd.DataFrame(np.random.standard_normal((2, 3)),
                   columns=["b", "d", "a"])
df1

,a,b,c,d
0,1.319266,0.851725,-1.274862,0.594504
1,-1.277035,1.366530,0.270048,0.215275
2,0.420139,0.330595,1.073252,1.217256


In [391]:
df2

,b,d,a
0,0.834263,2.224735,-0.378372
1,-0.075956,0.351265,-0.410430


In [393]:
# 传人ignore_index=True，丢弃了两个DataFrame的索引，只做数据拼接，并创建一个新索引
pd.concat([df1, df2], ignore_index=True)   #忽略索引，不按索引合并

,a,b,c,d
0,1.319266,0.851725,-1.274862,0.594504
1,-1.277035,1.366530,0.270048,0.215275
2,0.420139,0.330595,1.073252,1.217256
3,-0.378372,0.834263,NaN,2.224735
4,-0.410430,-0.075956,NaN,0.351265


忽略索引进行连接，并按照列名进行对齐，列名依然保持一致，一共5条信息，列一共有4列，分别是a、b、c、d，从df1到df2依次排序即可

![jupyter](8.12.png)

In [400]:
pd.concat([df2, df1], ignore_index=True)

,b,d,a,c
0,0.834263,2.224735,-0.378372,NaN
1,-0.075956,0.351265,-0.410430,NaN
2,0.851725,0.594504,1.319266,-1.274862
3,1.366530,0.215275,-1.277035,0.270048
4,0.330595,1.217256,0.420139,1.073252


与刚才df1在前有所不同，先是b,d,a最后再拼接c

拼接顺序：  
首先是 df2 的 2 行数据，然后接上 df1 的 3 行数据。  


列名对齐：  
由于 df2 只有 ["b", "d", "a"] 三列，而 df1 有 ["a", "b", "c", "d"] 四列，因此在 df2 中，c 列的值缺失，显示为 NaN。


索引重置：  
索引被忽略，生成新的整数索引 [0, 1, 2, 3, 4]。

In [407]:
pd.concat([df1, df2], axis=1, ignore_index=True)

,0,1,2,3,4,5,6
0,1.319266,0.851725,-1.274862,0.594504,0.834263,2.224735,-0.378372
1,-1.277035,1.366530,0.270048,0.215275,-0.075956,0.351265,-0.410430
2,0.420139,0.330595,1.073252,1.217256,NaN,NaN,NaN


忽略所有列名，df1有三行数据，先将df1的三行拼在前面，df2只有两行，所以第三行全为Na，结果会根据拼接顺序生成新的整数列名。

![jupyter](8.13.png)

#### pd.concat函数参数  

![jupyter](8.11.png)

#### 8.2.4 联合重叠数据  
还有一种数据联合场景不能用合并或拼接来处理。  
比如，有索引全部或部分重叠的两个数据集。  

In [418]:
a = pd.Series([np.nan, 2.5, 0.0, 3.5, 4.5, np.nan],
              index=["f", "e", "d", "c", "b", "a"])
b = pd.Series([0., np.nan, 2., np.nan, np.nan, 5.],
              index=["a", "b", "c", "d", "e", "f"])
a

f    NaN
e    2.5
d    0.0
c    3.5
b    4.5
a    NaN
dtype: float64

In [420]:
b

a    0.0
b    NaN
c    2.0
d    NaN
e    NaN
f    5.0
dtype: float64

In [422]:
np.where(pd.isna(a), b, a)

array([0. , 2.5, 0. , 3.5, 4.5, 5. ])

检查 a 中是否有 NaN 值。  
如果某个位置的值是 NaN，则使用 b 中对应位置的值 来替代。  
如果该位置的值不是 NaN，则保持 a 中原来的值。  

pd.isna(a)：检查 a 中是否存在 NaN，返回一个布尔型数组，True 表示该位置是 NaN。  
np.where(condition, x, y)：如果 condition 为 True，返回 x；否则返回 y。  

pd.isna(a)：  
检查 a 中是否存在 NaN，结果是 [False, True, True, True, False]。



np.where(pd.isna(a), b, a)：  
在第 1 和第 6 个位置 a 中有 NaN，所以从 b 中取值，对应的结果是 0. 和 5.。  
其他位置没有 NaN，所以保持 a 中原来的值  

In [436]:
# 使用numpy.where不会检查索引标签是否对齐（也不要求两个对象具有相同的长度）
# 所以如果想要按照索引排列值，需要使用Series的combine_first方法：
a.combine_first(b)     # 将a的Na值用b填入，并且索引对齐

a    0.0
b    4.5
c    3.5
d    0.0
e    2.5
f    5.0
dtype: float64

In [440]:
# 对于DataFrame, combine_first也会逐列做同样的操作，
# 因此可以认为用传入对象的数据为调用对象的缺失数据“打补丁”：
df1 = pd.DataFrame({"a": [1., np.nan, 5., np.nan],
                    "b": [np.nan, 2., np.nan, 6.],
                    "c": range(2, 18, 4)})
df2 = pd.DataFrame({"a": [5., 4., np.nan, 3., 7.],
                    "b": [np.nan, 3., 4., 6., 8.],})
df1

,a,b,c
0,1.0,NaN,2
1,NaN,2.0,6
2,5.0,NaN,10
3,NaN,6.0,14


In [442]:
df2

,a,b
0,5.0,NaN
1,4.0,3.0
2,NaN,4.0
3,3.0,6.0
4,7.0,8.0


In [444]:
df1.combine_first(df2)

,a,b,c
0,1.0,NaN,2.0
1,4.0,2.0,6.0
2,5.0,4.0,10.0
3,3.0,6.0,14.0
4,7.0,8.0,NaN


将df1中缺失的数据用df2取填补  
为什么df1没有第五行填补时却多了第五行？  


这是因为 combine_first() 方法在合并数据时，会根据 行索引对齐，并且如果 df2 中有 df1 中没有的索引，combine_first() 会将这些缺失的行添加到结果中。  


工作机制：  
对齐索引：combine_first() 首先检查 df1 和 df2 的索引。如果 df2 中有 df1 没有的行索引，combine_first() 会将这些行添加到结果中。   
填充缺失值：然后，combine_first() 会用 df2 中的值填充 df1 中的 NaN。   



combine_first() 不仅填补 NaN，还会把 df2 中 df1 没有的行（索引）加入到结果中。这使得最终的 DataFrame 包含所有唯一的行索引，并且在 df1 和 df2 中都找不到数据的地方会有 NaN。


![jupyter](8.14.png)

### 8.3 重塑和透视  
有多种基础操作用于重新排列表格型数据叫做重塑或透视  

首先要明白什么是stack(),它究竟在干嘛  


stack()函数就是将DF或Series中的 **列** 数据透视为 **行** 数据，将原先的多列（宽表），转化为多行（长表）。


In [208]:
# 举例：
example = pd.DataFrame({
    'A': [1, 2, 3],
    'B': [4, 5, 6],
    'C': [7, 8, 9]
}, index=['a', 'b', 'c'])
example

,A,B,C
a,1,4,7
b,2,5,8
c,3,6,9


In [210]:
# stack 操作：
example_stacked = example.stack()
example_stacked

a  A    1
   B    4
   C    7
b  A    2
   B    5
   C    8
c  A    3
   B    6
   C    9
dtype: int64

之前的行名(行索引) a b c 变为了行标签，并且成为了一级索引，之前的列名(列索引) A B C  变为了行索引，二级行索引， 因为之前的行索引每一行都会对应三个列，所以在列转变为行后，**每个行标签都对应三个列转换而来的行索引**，所以结果就生成了一个多级索引，并且列名都没了。


![jupyter](8.17.png)

In [215]:
# 利用 unstack()将其还原
example_unstacked = example_stacked.unstack()
example_unstacked

,A,B,C
a,1,4,7
b,2,5,8
c,3,6,9


stack 适用的场景：  


当需要将数据从 宽表格式 转换为 长表格式 时，stack 很有用。   



常用于数据预处理，特别是要对不同变量进行汇总分析时。  

#### 8.3.1 使用层次化索引进行重塑     
层次化索引为重排DF数据提供了一种具有良好一致性的方式，主要有两种操作：  


stack：  
    将数据的列“旋转”为行


unstack：  
    将数据的行透视为列。透视为列。

In [227]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(["Ohio", "Colorado"], name="state"),
                    columns=pd.Index(["one", "two", "three"],
                    name="number"))
data

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


行索引 (state): ["Ohio", "Colorado"]   
列索引 (number): ["one", "two", "three"]   

**这个 data 转换后和那个例子相同，将列 one two three 直接转化为二级行索引，共三列，之前的行索引变为行标签，每个标签三个二级索引(行索引)，行索引名为number**

In [229]:
# 对该数据使用stack方法即可将列透视为行，得到一个Series：
result = data.stack()
print(result)
pd.DataFrame(result)   #多个0是因为外层列没有列名，所以用0填补了

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int32


0
state    number   
Ohio     one     0
         two     1
         three   2
Colorado one     3
         two     4
         three   5

stack堆叠后，将列索引number堆叠到行索引上，此时行索引是两级索引(state, number)，而值还是原来的值，原来DF中的数据，这时的number成为了多级索引的第二层

data.stack() 的作用是将 列索引 堆叠为 行索引，将原本的宽表转换为长表。  
data 是一个具有 单级行索引（state）和 单级列索引（number）的 DataFrame。  

stack() 操作：将原本的 列索引 ["one", "two", "three"] 转换为行索引的第二级索引。  

输出的 result 是一个 多级索引的 Series，其中第一层索引是 state，第二层索引是原来的列 number。  

行索引：由两级组成，第一层是 state（Ohio 和 Colorado），第二层是 number（one，two，three）。  

stack() 将 DataFrame 中的 列 堆叠为 行索引 的一部分，生成了一个长表形式的 Series 数据。  
原本的列数据被折叠成行数据，行索引变为多级索引，方便对数据的进一步处理和分析。  

In [461]:
data.T

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


stack() 与 .T 区别  
功能不同：  

stack() 是用于将 列 堆叠到 行，将 DataFrame 从宽表转换为长表。  
T 是用于 转置 DataFrame，将行和列互换。  


结果形态不同：  

stack() 产生一个 多级索引的长格式数据。  
T 仅是简单的 行列互换，没有改变表的结构。  

In [231]:
# 对于一个层次化索引的Series，你可以用unstack方法将其重排为DataFrame：
result.unstack()

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


**一般情况下，unstack是将行的内层索引(默认)转化为列，如果指定了层级，则会按照该层级进行展开**

In [233]:
# 默认情况下，unstack操作的是最内层（stack也是如此）。
# 传入层级编号或名称，即可对其他层级进行unstack操作：
result.unstack(level=0)    # 将最外层行索引转化为列

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


**将第一级行索引 state 转换为 列索引**，并将数据的格式重新展开为宽格式。 


因为 Ohio 和 Colorado 都是行标签，所以从行标签到列索引，Ohio 和 Colorado 都是对应三行内容，原先的二级行索引分别是 Ohio 对应 one two three，Colorado 对应 one two three，
层级越在外面层级越高，因为只是将state层级的列恢复到宽格式，所以number 列仍然还在。  


新 DataFrame 的 列索引 是 Ohio 和 Colorado，行索引 是原 DataFrame 的列 number (one, two, three)，这些行索引中的每一行代表了不同的 number 值。

![jupyter](8.15.png)

In [243]:
result.unstack(level="state")

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


level="state" 将名称为 state 的索引层（即 Ohio 和 Colorado）从 行索引 转换为 列索引。  
与 unstack(level=0) 的结果是一样的，因为 state 是第一级行索引（level=0）  

**目前都是索引标签对应二级索引行个数相同的情况，比如 Ohio 对应 one two three， Colorado 也是对应 one two three 三行，如果遇到行数不对等的情况呢？**

当执行 unstack() 操作时，如果单层索引数量不一致，会出现 缺失值 (NaN) 的情况。



这是因为 unstack() 操作会将索引值转化为列标签，并且按原始索引的所有可能组合来排列数据。



如果某些组合不存在，则填充为 NaN。

In [270]:
example2 = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6]}, index=[['x', 'x', 'y'], ['a', 'b', 'a']])
example2

A  B
x a  1  4
  b  2  5
y a  3  6

In [272]:
# 进行转化
unstack_example2 = example2.unstack(level=0)
unstack_example2

A         B     
     x    y    x    y
a  1.0  3.0  4.0  6.0
b  2.0  NaN  5.0  NaN

这个例子中，索引标签 y 缺少 b 行，所以在转化后的 DF 上填充了Na值

**新问题：为什么上面的 Ohio 和 Colorado 仅仅只出现了一次作为列标签，这个x、y为什么出现了两次？**

因为 Ohio Colorado 那个例子，原数据仅有一列，所以即使从行索引转化为列索引，也就那一列数据需要列索引，但x、y这个例子，它原本就有两列数据，分别还有列标签 A B，所以在进行转换的时候会生成两次x、y的组合，以对应两列的数据。

In [241]:
result.unstack(level="number")   #等同于level=1 第二层

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


level=number 只对number列(行索引)进行转化(转换为列索引)，将number列由列转为行，变为列索引，由长格式转换为宽格式。  

![jupyter](8.16.png)

In [500]:
#  如果在各子分组中不能找到所有层级的值，则unstack操作可能会导入缺失数据：
s1 = pd.Series([0, 1, 2, 3], index=["a", "b", "c", "d"], dtype="Int64")
s2 = pd.Series([4, 5, 6], index=["c", "d", "e"], dtype="Int64")
data2 = pd.concat([s1, s2], keys=["one", "two"])
data2

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: Int64

In [506]:
# stack操作会默认过滤缺失数据，因此该运算是可逆的：
data2.unstack()

,a,b,c,d,e
one,0,1,2,3,<NA>
two,<NA>,<NA>,4,5,6


In [516]:
data2.unstack().stack()

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: Int64

当有 NaN/缺失值、索引不唯一、索引层级丢失或改变 时，unstack() 可能是不可逆的。

In [511]:
data2.unstack().stack(dropna=False)

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_2624\3936770077.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data2.unstack().stack(dropna=False)


one  a       0
     b       1
     c       2
     d       3
     e    <NA>
two  a    <NA>
     b    <NA>
     c       4
     d       5
     e       6
dtype: Int64

这个警告信息是因为你正在使用的 Pandas 版本引入了对 stack() 方法的新实现，而当前的 stack() 方法即将被废弃。在未来的版本中，Pandas 将会移除旧的 stack() 实现。

In [522]:
# 去除dropna=False，因为未来的future_stack=True 默认不删除Na值
data2.unstack().stack(future_stack=True)  

one  a       0
     b       1
     c       2
     d       3
     e    <NA>
two  a    <NA>
     b    <NA>
     c       4
     d       5
     e       6
dtype: Int64

In [526]:
result

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int32

In [283]:
# 在对DataFrame进行unstack操作时，被拆分的层级将成为结果中的最低层级：
df = pd.DataFrame({"left": result, "right": result + 5},
                  columns=pd.Index(["left", "right"], name="side"))
df

side             left  right
state    number             
Ohio     one        0      5
         two        1      6
         three      2      7
Colorado one        3      8
         two        4      9
         three      5     10

"left": result：  
将 result 中的值作为 left 列。它保留了 result 的索引结构（state 和 number 的多级索引），所以 DataFrame 的行索引仍然是原来的多级索引。  


"right": result + 5：  
result + 5 将 result 中的每个值加上 5，作为 right 列。result 是一个 Series，所以加法会作用在每个元素上。它的索引结构也和 result 一样，所以它将成为 right 列中的数据。


columns=pd.Index(["left", "right"], name="side")：  
这里你通过 pd.Index() 创建了一个命名的列索引，列名为 "left" 和 "right"，索引名称为 "side"。所以生成的 DataFrame 会有一个名为 "side" 的列层次索引，其中包括 "left" 和 "right" 两个子列。

In [285]:
df.unstack(level="state")

side   left          right         
state  Ohio Colorado  Ohio Colorado
number                             
one       0        3     5        8
two       1        4     6        9
three     2        5     7       10

**此例子和刚才的example2相似，因为有两列 left 和 right，所以最后当 state 的 Ohio 和 Colorado 由行标签转换为列索引时，在原先的每一列生成列索引名称，和数据完美匹配**

In [287]:
# 与unstack操作相同，当调用stack时，可以指明轴的名称：
df.unstack(level="state").stack(level="side")

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_6232\4202836905.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.unstack(level="state").stack(level="side")


state         Ohio  Colorado
number side                 
one    left      0         3
       right     5         8
two    left      1         4
       right     6         9
three  left      2         5
       right     7        10

这个警告信息来自于 pandas 库中即将弃用的 stack 方法。你正在使用的是旧版的 stack 实现，而 pandas 2.1.0 中引入了新的 stack 实现，旧的实现会在未来的版本中被移除。

In [289]:
# 无论是转换后的行还是列索引，都在最内层作为索引存在
df.unstack(level="state").stack(level="side", future_stack=True)

state         Ohio  Colorado
number side                 
one    left      0         3
       right     5         8
two    left      1         4
       right     6         9
three  left      2         5
       right     7        10

对于side 有两个列标签，要对列标签进行转换，无论是怎么转换，被转换的那个都要进入到最内层，所以 side 转换后在one two three 行标签里面， 且，因为一开始有 one two three 三个行标签，left 和 right 转换后会生成三组行索引去匹配 one two three，最后一共有 6 个行索引生成


![jupyter](8.18.png)

#### 8.3.2 将 "长格式" 透视为 "宽格式"  
多个时间序列数据通常是以所谓的“长格式”或“堆叠格式”存储在数据库和CSV中的。  
对于这种格式，表中的每行表示单个值，而不是表示多个值。  

In [12]:
# 时间序列规整和数据清洗
data = pd.read_csv("../examples/macrodata.csv")
data = data.loc[:,["year", "quarter", "realgdp", "infl", "unemp"]]
data.head()

,year,quarter,realgdp,infl,unemp
0,1959,1,2710.349,0.00,5.8
1,1959,2,2778.801,2.34,5.1
2,1959,3,2775.488,2.74,5.3
3,1959,4,2785.204,0.27,5.6
4,1960,1,2847.699,2.31,5.2


In [8]:
# 使用pandas.PeriodIndex（来表示时间区间，而非时间点）将year和quarter列联合并设置索引，
# 让索引包含每个季度末的datetime值。
periods = pd.PeriodIndex(year=data.pop("year"),
                         quarter=data.pop("quarter"),
                         name="date")
periods

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_14900\485839322.py:3: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  periods = pd.PeriodIndex(year=data.pop("year"),


PeriodIndex(['1959Q1', '1959Q2', '1959Q3', '1959Q4', '1960Q1', '1960Q2',
             '1960Q3', '1960Q4', '1961Q1', '1961Q2',
             ...
             '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3',
             '2008Q4', '2009Q1', '2009Q2', '2009Q3'],
            dtype='period[Q-DEC]', name='date', length=203)

这个警告信息表明，PeriodIndex 的构造方法已经被弃用，建议使用 PeriodIndex.from_fields 方法。

创建一个 Pandas 中的 PeriodIndex ，它是一个专门用于表示时间段的索引。使用 data 中的 year 和 quarter 列来构建这个 PeriodIndex。  

内容的细分：  

时间段表示：PeriodIndex 中的每个条目是年份和季度的组合（例如，'1959Q1' 表示1959年的第一季度）。  

dtype='period[Q-DEC]'：这表示这些时间段是以季度为单位，并且以12月作为年份的结束。  

length=203：这表明 PeriodIndex 包含203个时间段，从1959年的第一季度到2009年的第三季度。  

In [14]:
periods = pd.PeriodIndex.from_fields(year=data.pop("year"),
                                     quarter=data.pop("quarter"),
                                     #name="date"
                                    )

# 更新后的fields不接受name参数了，所以只能构造后再赋值
periods.name = "date"
periods

PeriodIndex(['1959Q1', '1959Q2', '1959Q3', '1959Q4', '1960Q1', '1960Q2',
             '1960Q3', '1960Q4', '1961Q1', '1961Q2',
             ...
             '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3',
             '2008Q4', '2009Q1', '2009Q2', '2009Q3'],
            dtype='period[Q-DEC]', name='date', length=203)

PeriodIndex.from_fields 是 Pandas 中用于创建 PeriodIndex 的一个方法，允许你通过分别提供年份、季度、月份等字段来生成时间段索引。  
year: 一个包含年份的数组或列表。  
quarter: 一个包含季度的数组或列表（值应为 1 到 4）。   
你也可以传入其他参数，如月份、日等，以创建更精确的时间段。   

PeriodIndex.from_fields() 方法不接受 name 参数。你可以先构造 PeriodIndex，然后再给它命名。

In [16]:
# 如果没有天，默认是1    将data的index设置为年份和季度合并的格式
data.index = periods.to_timestamp("D")    # D 是天级别的精度
data.head()

,realgdp,infl,unemp
date,,,
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


pop方法会将弹出值的DF清空

In [18]:
# 选取部分列，将columns 的索引名设为 item
data = data.reindex(columns=["realgdp", "infl", "unemp"])
data.columns.name = "item"
data.head()

item,realgdp,infl,unemp
date,,,
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


为什么列名都在上面？  
因为第一列是data的index，索引的name就是data，如果没有列名，那么这三列就不会再上面。

In [41]:
# 详细解析
# 先stack操作  将多列转化为多行，将除了时间列都转为行
long_data = pd.DataFrame((data.stack()))
long_data[:10]

0
date       item             
1959-01-01 realgdp  2710.349
           infl        0.000
           unemp       5.800
1959-04-01 realgdp  2778.801
           infl        2.340
           unemp       5.100
1959-07-01 realgdp  2775.488
           infl        2.740
           unemp       5.300
1959-10-01 realgdp  2785.204

将多列转化为多行  

![jupyter](8.19.png)

In [43]:
#重置索引，原先的索引列将会变为二级索引，一级索引是重新排序的默认值索引
long_data = long_data.reset_index()  
long_data[:10]

,date,item,0
0,1959-01-01,realgdp,2710.349
1,1959-01-01,infl,0.000
2,1959-01-01,unemp,5.800
3,1959-04-01,realgdp,2778.801
4,1959-04-01,infl,2.340
5,1959-04-01,unemp,5.100
6,1959-07-01,realgdp,2775.488
7,1959-07-01,infl,2.740
8,1959-07-01,unemp,5.300
9,1959-10-01,realgdp,2785.204


In [49]:
# 将列名为 0 的列重命名为 value
long_data = long_data.rename(columns={0:"value"})
long_data[:10]

,date,item,value
0,1959-01-01,realgdp,2710.349
1,1959-01-01,infl,0.000
2,1959-01-01,unemp,5.800
3,1959-04-01,realgdp,2778.801
4,1959-04-01,infl,2.340
5,1959-04-01,unemp,5.100
6,1959-07-01,realgdp,2775.488
7,1959-07-01,infl,2.740
8,1959-07-01,unemp,5.300
9,1959-10-01,realgdp,2785.204


以上三步操作就是下面那一列的解析，同等结果

In [332]:
# 使用stack操作进行重塑，并使用reset_index将新的索引层级移到列
# 最后将包含数据值的列命名为"value"：
long_data = (data.stack()   #列转行
             .reset_index()    #层次化索引转变普通列
             .rename(columns={0: "value"}))      #堆叠后的值列重命名
long_data[:10]

,date,item,value
0,1959-01-01,realgdp,2710.349
1,1959-01-01,infl,0.000
2,1959-01-01,unemp,5.800
3,1959-04-01,realgdp,2778.801
4,1959-04-01,infl,2.340
5,1959-04-01,unemp,5.100
6,1959-07-01,realgdp,2775.488
7,1959-07-01,infl,2.740
8,1959-07-01,unemp,5.300
9,1959-10-01,realgdp,2785.204


**虽然我看不懂，但我大为震惊**

这就是包含多个时间序列的长格式，表中的每一行代表一次观测。

关系型数据库中的数据通常是这样存储的，这是因为随着数据添不断加到数据库中，固定模式（即列名和数据类型）可使item列中不同值的数量随之改变。


在前面的例子中，date和item通常作为主键（使用关系型数据库的说法），不仅提供了关系完整性，而且使连接更为简单。但在某些情况下，使用此类格式的数据会很棘手。


你可能更偏向DataFrame，让不同的item值独立包含一列，使用date列中的时间戳作为索引。


DataFrame的pivot方法就是用来实现此种转换的：

细究pivot方法：   **pivot操作是stack的逆操作**   
pivot 方法用于将长格式数据转换为宽格式数据，主要目的是让数据更易于分析和可视化。  
主要用途  
数据重组: 将多行数据转换为更紧凑的形式，使每个独立的分类（如 item）在自己的列中。  
更直观的分析: 宽格式数据可以更清晰地显示不同分类在特定时间点的值，便于比较和分析。  
简化可视化: 在绘图时，宽格式通常更容易与图表工具配合使用，比如时间序列图。  


有一个销售数据集，每一天的不同产品的销售量。长格式的数据可能像这样：  

![jupyter](8.20.png)


使用 pivot 后，可以将其转换为宽格式：  


![jupyter](8.21.png)

pivot 使得每个 item 的值变成单独的列，便于观察每个产品在不同日期的销售情况，从而可以更方便地进行数据分析和可视化。  

In [340]:
#以data为索引，将item列拆开，item中的列值从value中提取
pivoted = long_data.pivot(index="date", columns="item",
                          values="value")
pivoted.head()

item,infl,realgdp,unemp
date,,,
1959-01-01,0.00,2710.349,5.8
1959-04-01,2.34,2778.801,5.1
1959-07-01,2.74,2775.488,5.3
1959-10-01,0.27,2785.204,5.6
1960-01-01,2.31,2847.699,5.2


将item中的不同值转化为单个列，便于统计和可视化，填充值则是从value列中来


index="date": 将 date 列作为新的索引，这样每一行将代表一个唯一的日期。  
columns="item": 将 item 列中的不同值转换为新的列，每个独立的 item 值会成为 DataFrame 的一列。  
values="value": 用于填充新表格的值来自于 value 列，具体来说，每个日期和每个 item 的组合对应的值。  



前两个传入的列分别用作行索引和列索引，最后一个可选项则是用于填充DataFrame的
数值列。

In [68]:
# 假设有两个需要同时重塑的数值列：
long_data["value2"] = np.random.standard_normal(len(long_data))
long_data[:10]

,date,item,value,value2
0,1959-01-01,realgdp,2710.349,0.276593
1,1959-01-01,infl,0.000,-0.023636
2,1959-01-01,unemp,5.800,1.106907
3,1959-04-01,realgdp,2778.801,-0.047432
4,1959-04-01,infl,2.340,-1.066849
5,1959-04-01,unemp,5.100,-1.665186
6,1959-07-01,realgdp,2775.488,0.113657
7,1959-07-01,infl,2.740,-2.388996
8,1959-07-01,unemp,5.300,-1.203731
9,1959-10-01,realgdp,2785.204,0.684318


In [82]:
test = long_data.pivot(index="date", columns="item", values=["value", "value2"])
test[:10]

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-01-01  0.00  2710.349   5.8 -0.023636  0.276593  1.106907
1959-04-01  2.34  2778.801   5.1 -1.066849 -0.047432 -1.665186
1959-07-01  2.74  2775.488   5.3 -2.388996  0.113657 -1.203731
1959-10-01  0.27  2785.204   5.6 -0.214751  0.684318  0.503023
1960-01-01  2.31  2847.699   5.2 -0.235285 -1.477788  0.372311
1960-04-01  0.14  2834.390   5.2 -0.767509 -0.272474  0.560415
1960-07-01  2.70  2839.022   5.6 -1.098439 -0.525027 -1.467541
1960-10-01  1.21  2802.616   6.3  1.183301 -0.173567 -0.987181
1961-01-01 -0.40  2819.264   6.8 -0.973049 -0.941624 -1.452586
1961-04-01  1.47  2872.005   7.0  0.816116 -1.530477  0.002457

In [72]:
# 如果忽略最后一个参数，得到的DataFrame就会带有层次化的列：
pivoted = long_data.pivot(index="date", columns="item")
pivoted.head()

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-01-01  0.00  2710.349   5.8 -0.023636  0.276593  1.106907
1959-04-01  2.34  2778.801   5.1 -1.066849 -0.047432 -1.665186
1959-07-01  2.74  2775.488   5.3 -2.388996  0.113657 -1.203731
1959-10-01  0.27  2785.204   5.6 -0.214751  0.684318  0.503023
1960-01-01  2.31  2847.699   5.2 -0.235285 -1.477788  0.372311

long_data.pivot(index="date", columns="item", values=["value", "value2"])  
long_data.pivot(index="date", columns="item")  
两种写法效果相同

In [91]:
# 选取具体列索引的数据查看
test["value2"].head()

item,infl,realgdp,unemp
date,,,
1959-01-01,-0.023636,0.276593,1.106907
1959-04-01,-1.066849,-0.047432,-1.665186
1959-07-01,-2.388996,0.113657,-1.203731
1959-10-01,-0.214751,0.684318,0.503023
1960-01-01,-0.235285,-1.477788,0.372311


In [348]:
pivoted["value"].head()

item,infl,realgdp,unemp
date,,,
1959-01-01,0.00,2710.349,5.8
1959-04-01,2.34,2778.801,5.1
1959-07-01,2.74,2775.488,5.3
1959-10-01,0.27,2785.204,5.6
1960-01-01,2.31,2847.699,5.2


**注意，pivot其实就等价于先用set_index创建层次化索引，再用unstack重塑：**

In [351]:
unstacked = long_data.set_index(["date", "item"]).unstack(level="item")
unstacked.head()

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-01-01  0.00  2710.349   5.8 -0.899957 -1.107473  2.029231
1959-04-01  2.34  2778.801   5.1  0.582391  1.305941  1.133188
1959-07-01  2.74  2775.488   5.3 -1.065632  0.220678 -1.567512
1959-10-01  0.27  2785.204   5.6  0.350381  2.537366 -0.582628
1960-01-01  2.31  2847.699   5.2  0.624016 -0.635318 -0.399247

#### 8.3.3 将 "宽格式" 透视为 "长格式"  
对于DataFrame, pivot操作的逆运算是pandas.melt。它不是将一列转换为新DataFrame中的多列，而是将多个列合并成一列，并生成比输入更长的DataFrame

In [357]:
df = pd.DataFrame({"key": ["foo", "bar", "baz"],
                   "A": [1, 2, 3],
                   "B": [4, 5, 6],
                   "C": [7, 8, 9]})
df

,key,A,B,C
0,foo,1,4,7
1,bar,2,5,8
2,baz,3,6,9


"key"列可以作为分组指标，其他列用作数据值。  





当使用pandas.melt时，必须指明哪些列（如果有的话）是分组指标。  

In [361]:
# 使用"key”作为唯一的分组指标：
# 宽(每个分类量一列)转长(所有变量全在一列)
melted = pd.melt(df, id_vars="key")
melted

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6
6,foo,C,7
7,bar,C,8
8,baz,C,9


In [363]:
# 使用 pivot 将其重塑回原来的形式;
reshaped = melted.pivot(index="key", columns="variable",
                        values="value")
reshaped

variable,A,B,C
key,,,
bar,2,5,8
baz,3,6,9
foo,1,4,7


因为pivot的结果是从列创建了一个索引，并用作行标签，所以我们可以使用reset_index将数据再移回到列：

In [366]:
reshaped.reset_index()

variable,key,A,B,C
0,bar,2,5,8
1,baz,3,6,9
2,foo,1,4,7


In [368]:
# 指定列的子集用作 value 列：
pd.melt(df, id_vars="key", value_vars=["A", "B"])

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6


In [370]:
# pd.melt 也可以不用任意分组指标：
pd.melt(df, value_vars=["A", "B", "C"])

,variable,value
0,A,1
1,A,2
2,A,3
3,B,4
4,B,5
5,B,6
6,C,7
7,C,8
8,C,9


In [372]:
pd.melt(df, value_vars=["key", "A", "B"])

,variable,value
0,key,foo
1,key,bar
2,key,baz
3,A,1
4,A,2
5,A,3
6,B,4
7,B,5
8,B,6
